In [1]:
from QLBM import QLBMV2, collision, InitializeQC
import numpy as np
import matplotlib.pyplot as plt
import qiskit_aer
from qiskit import transpile

In [2]:
# Domain and grid setup
N_POINTS_X, N_POINTS_Y, N_POINTS_Z = 8, 8, 8
# Simulation parameters

NUMBER_DISCRETE_VELOCITIES = 27  # D2Q9 lattice configuration

In [3]:
Qq = 27
Nx = N_POINTS_X-1
Ny = N_POINTS_Y-1
Nz = N_POINTS_Y-1
dx = 1
dy = 1
dz = 1
dt = dx
c = dt/dx
Re = 10
f_eq = np.zeros((Qq, Nx+1, Ny+1, Ny+1))
f = np.zeros((Qq, Nx+1, Ny+1, Ny+1))
f_star = np.zeros((Qq, Nx+1, Ny+1, Ny+1))
Lx = dx * float(Nx+1)/2.0
Ly = dy * float(Ny+1)/2.0
Lz = dz * float(Nz+1)/2.0
U = 0.2
nu = U*Lx/Re

cs = np.sqrt(c**2/3)
tau_f = 1.0
rho_0 = 1.0
rho = np.zeros((Nx+1, Ny+1, Ny+1))
u = np.zeros((Nx+1, Ny+1, Ny+1, 3))
e = np.array([
    [0,  1, -1,  0,  0,   0,  0,  1, -1,  1, -1,  0,  0,  1, -1,  1, -1,  0,  0,  1, -1,  1, -1,  1, -1, -1,  1],
    [0,  0,  0,  1, -1,   0,  0,  1, -1,  0,  0,  1, -1, -1,  1,  0,  0,  1, -1,  1, -1,  1, -1, -1,  1,  1, -1],
    [0,  0,  0,  0,  0,   1, -1,  0,  0,  1, -1,  1, -1,  0,  0, -1,  1, -1,  1,  1, -1, -1,  1,  1, -1,  1, -1]
]).T
w = np.array([
    8/27,                   
    2/27, 2/27, 2/27, 2/27, 2/27, 2/27,  
    1/54, 1/54, 1/54, 1/54, 1/54, 1/54, 1/54, 1/54, 1/54, 1/54, 1/54, 1/54,  
    1/216, 1/216, 1/216, 1/216, 1/216, 1/216, 1/216, 1/216  
])

A = 3/4 - 9/2*U*Lx/(Re*dt)
nu1 = (1/6-2/9*A)*dt
u_n = u[:, :, :, 0].copy()
v_n = u[:, :, :, 1].copy() 
w_n = u[:, :, :, 2].copy()
u_t = np.zeros((Nx+3, Ny+3, Ny+3))
v_t = np.zeros((Nx+3, Ny+3, Ny+3))
w_t = np.zeros((Nx+3, Ny+3, Ny+3))
t_s = 1.0
TIMESTEPS = int(t_s*Lx/U)
print(nu,nu1,Lx,Ly,TIMESTEPS)

q_error=[]
c_error=[]
qc_error=[]

0.08 0.07999999999999999 4.0 4.0 20


In [4]:
## Initial
x0 = np.linspace(-Lx+dx/2,Lx-dx/2,Nx+1)
y0 = np.linspace(-Ly+dy/2,Ly-dy/2,Ny+1)
z0 = np.linspace(-Lz+dz/2,Lz-dz/2,Nz+1)
print(x0,y0,z0)
X0,Y0,Z0 = np.meshgrid(x0,y0,z0)
rho[:, :, :] = rho_0
u[:, :, :, 0] = -U*np.cos(np.pi*X0/Lx)*np.sin(np.pi*Y0/Ly)
u[:, :, :, 1] = U*np.sin(np.pi*X0/Lx)*np.cos(np.pi*Y0/Ly)
u[:, :, :, 2] = 0

[-3.5 -2.5 -1.5 -0.5  0.5  1.5  2.5  3.5] [-3.5 -2.5 -1.5 -0.5  0.5  1.5  2.5  3.5] [-3.5 -2.5 -1.5 -0.5  0.5  1.5  2.5  3.5]


In [5]:
simulator = qiskit_aer.backends.statevector_simulator.StatevectorSimulator()

In [6]:
# Initialize the quantum LBM scalar field
Psi_qlbm = np.zeros((TIMESTEPS + 1, N_POINTS_X, N_POINTS_Y, N_POINTS_Z))
Psi_qlbm1 = np.zeros((TIMESTEPS + 1, N_POINTS_X, N_POINTS_Y, N_POINTS_Z))
Psi_qlbm2 = np.zeros((TIMESTEPS + 1, N_POINTS_X, N_POINTS_Y, N_POINTS_Z))
Psi_qlbm3 = np.zeros((TIMESTEPS + 1, N_POINTS_X, N_POINTS_Y, N_POINTS_Z))
Psi_qlbm[0, :, :, :] = rho_0#Psi_init
Psi_qlbm1[0, :, :, :] = u[:,:,:,0].copy()#Psi_init
Psi_qlbm2[0, :, :, :] = u[:,:,:,1].copy()#Psi_init
Psi_qlbm3[0, :, :, :] = u[:,:,:,2].copy()#Psi_init
Psi_qlbm0 = Psi_qlbm[0,:,:,:].copy()
u_LBM = np.zeros((N_POINTS_X, N_POINTS_Y, N_POINTS_Z, 3))
u_LBM[:, :, :, 0] = Psi_qlbm1[0, :, :,:]  # Set the x-component of the velocity
u_LBM[:, :, :, 1] = Psi_qlbm2[0, :, :,:]  # Set the y-component of the velocity
u_LBM[:, :, :, 2] = Psi_qlbm3[0, :, :,:]  # Set the z-component of the velocity

# Quantum LBM simulation loop
for t in range(TIMESTEPS):
    u_t[1:-1,1:-1,1:-1] = u[:,:,:,0].copy()
    v_t[1:-1,1:-1,1:-1] = u[:,:,:,1].copy()
    w_t[1:-1,1:-1,1:-1] = u[:,:,:,2].copy()

    u_t[0,1:-1,1:-1] = u[-1,:,:,0].copy()
    u_t[-1,1:-1,1:-1] = u[0,:,:,0].copy()
    u_t[1:-1,0,1:-1] = u[:,-1,:,0].copy()
    u_t[1:-1,-1,1:-1] = u[:,0,:,0].copy()
    u_t[1:-1,1:-1,0] = u[:,:,-1,0].copy()
    u_t[1:-1,1:-1,-1] = u[:,:,0,0].copy()

    v_t[0,1:-1,1:-1] = u[-1,:,:,1].copy()
    v_t[-1,1:-1,1:-1] = u[0,:,:,1].copy()
    v_t[1:-1,0,1:-1] = u[:,-1,:,1].copy()
    v_t[1:-1,-1,1:-1] = u[:,0,:,1].copy()
    v_t[1:-1,1:-1,0] = u[:,:,-1,1].copy()
    v_t[1:-1,1:-1,-1] = u[:,:,0,1].copy()

    w_t[0,1:-1,1:-1] = u[-1,:,:,2].copy()
    w_t[-1,1:-1,1:-1] = u[0,:,:,2].copy()
    w_t[1:-1,0,1:-1] = u[:,-1,:,2].copy()
    w_t[1:-1,-1,1:-1] = u[:,0,:,2].copy()
    w_t[1:-1,1:-1,0] = u[:,:,-1,2].copy()
    w_t[1:-1,1:-1,-1] = u[:,:,0,2].copy()


    u_t_y = (u_t[1:-1, 2:, 1:-1] - u_t[1:-1, :-2, 1:-1])/(2.0*dy)
    u_t_x = (u_t[2:, 1:-1, 1:-1] - u_t[:-2, 1:-1, 1:-1])/(2.0*dx)
    u_t_z = (u_t[1:-1, 1:-1, 2:] - u_t[1:-1, 1:-1, :-2])/(2.0*dz)
   
    v_t_y = (v_t[1:-1, 2:, 1:-1] - v_t[1:-1, :-2, 1:-1])/(2.0*dy)
    v_t_x = (v_t[2:, 1:-1, 1:-1] - v_t[:-2, 1:-1, 1:-1])/(2.0*dx)
    v_t_z = (v_t[1:-1, 1:-1, 2:] - v_t[1:-1, 1:-1, :-2])/(2.0*dz)

    w_t_y = (w_t[1:-1, 2:, 1:-1] - w_t[1:-1, :-2, 1:-1])/(2.0*dy)
    w_t_x = (w_t[2:, 1:-1, 1:-1] - w_t[:-2, 1:-1, 1:-1])/(2.0*dx)
    w_t_z = (w_t[1:-1, 1:-1, 2:] - w_t[1:-1, 1:-1, :-2])/(2.0*dz)


    temp = u[:, :, :, 0] * u[:, :, :, 0] + u[:, :, :, 1] * u[:, :, :, 1]+ u[:, :, :, 2] * u[:, :, :, 2]
    for Q in range(27):
        f_eq[Q,:, :, :] = w[Q]*rho*(1.0 + 3.0 * (e[Q, 0] * u[:, :, :, 0] + e[Q, 1] * u[:, :, :, 1] + e[Q, 2] * u[:, :, :, 2]) + 4.5 * (e[Q, 0] * u[:, :, :, 0] + e[Q, 1] * u[:, :, :, 1] + e[Q, 2] * u[:, :, :, 2]) ** 2 - 1.5 * temp +\
                                  A*dt*(u_t_x + u_t_x)*e[Q,0]*e[Q,0] + A*dt*(v_t_x + u_t_y)*e[Q,0]*e[Q,1] + A*dt*(w_t_x + u_t_z)*e[Q,0]*e[Q,2]+\
                                  A*dt*(u_t_y + v_t_x)*e[Q,0]*e[Q,1] + A*dt*(v_t_y + v_t_y)*e[Q,1]*e[Q,1] + A*dt*(w_t_y + v_t_z)*e[Q,2]*e[Q,1]+\
                                  A*dt*(u_t_z + w_t_x)*e[Q,0]*e[Q,2] + A*dt*(v_t_z + w_t_y)*e[Q,1]*e[Q,2] + A*dt*(w_t_z + w_t_z)*e[Q,2]*e[Q,2]
                                  )
        f[Q, :, :, :] = f[Q, :, :, :] - (f[Q, :, :, :] - f_eq[Q, :, :, :]) / tau_f

    for vvv in range(27):
        f[vvv, :, :, :] = np.roll(
            np.roll(
                np.roll(f[vvv, :, :, :], e[vvv, 0], axis=0),
                e[vvv, 1], axis=1
            ),
            e[vvv, 2], axis=2
        )

    rho = np.sum(f, axis=0)
    u[:, :, :, 0] = np.sum(f * e[:,0].reshape(27,1,1,1), axis=0) / rho
    u[:, :, :, 1] = np.sum(f * e[:,1].reshape(27,1,1,1), axis=0) / rho
    u[:, :, :, 2] = np.sum(f * e[:,2].reshape(27,1,1,1), axis=0) / rho

    error = np.sum(np.sqrt((u_n-u[:, :, :, 0])**2+(v_n-u[:, :, :, 1])**2+(w_n-u[:, :, :, 2])**2))/np.sum(np.sqrt(u[:, :, :, 0]**2+u[:, :, :, 1]**2+u[:, :, :, 2]**2))
    u_n = u[:, :, :, 0].copy() 
    v_n = u[:, :, :, 1].copy()
    w_n = u[:, :, :, 2].copy()

    # Create and run the quantum circuit for LBM, rho
    qc = QLBMV2(density_field=Psi_qlbm[t, :, :, :], velocity_field=u_LBM, number_velocities=NUMBER_DISCRETE_VELOCITIES)
    
    # Compile and run the quantum circuit
    compiled_circuit = transpile(qc, simulator)
    result = simulator.run(compiled_circuit).result()
    
    # Extract and process the statevector
    statevector = np.array(result.get_statevector())
    real_part_statevector = np.real(statevector[:N_POINTS_X * N_POINTS_Y * N_POINTS_Z*27])
    reshaped_real_part = np.reshape(real_part_statevector, (N_POINTS_X, N_POINTS_Y, N_POINTS_Z,27),order='F')
    
    # Normalize and update Psi_qlbm
    norm_factor = np.linalg.norm(Psi_qlbm[t, :, :, :].flatten())
    fq = reshaped_real_part * norm_factor
    Psi_qlbm[t + 1, :, :, :] = np.sum(fq, axis=3)
    Psi_qlbm1[t + 1, :, :, :] = np.sum(fq * e[:,0].reshape(1,1,1,27), axis=3) / Psi_qlbm[t + 1, :, :, :]
    Psi_qlbm2[t + 1, :, :, :] = np.sum(fq * e[:,1].reshape(1,1,1,27), axis=3) / Psi_qlbm[t + 1, :, :, :]
    Psi_qlbm3[t + 1, :, :, :] = np.sum(fq * e[:,2].reshape(1,1,1,27), axis=3) / Psi_qlbm[t + 1, :, :, :]
    
    Psi_qlbm0 = Psi_qlbm[t + 1, :, :, :].copy()
    
    u_LBM[:,:, :,0] = Psi_qlbm1[t + 1, :, :, :]
    u_LBM[:,:, :,1] = Psi_qlbm2[t + 1, :, :, :]
    u_LBM[:,:, :,2] = Psi_qlbm3[t + 1, :, :, :]
    error1 = np.sum(np.sqrt((Psi_qlbm1[t + 1, :, :, :]-Psi_qlbm1[t, :, :, :])**2+(Psi_qlbm2[t + 1, :, :, :]-Psi_qlbm2[t, :, :, :])**2+(Psi_qlbm3[t + 1, :, :, :]-Psi_qlbm3[t, :, :, :])**2))/np.sum(np.sqrt(Psi_qlbm1[t + 1, :, :, :]**2+Psi_qlbm2[t + 1, :, :, :]**2+Psi_qlbm3[t + 1, :, :, :]**2))
    error2 = np.sum(np.sqrt((Psi_qlbm1[t + 1, :, :, :]-u_n)**2+(Psi_qlbm2[t + 1, :, :, :]-v_n)**2+(Psi_qlbm3[t + 1, :, :, :]-w_n)**2))/np.sum(np.sqrt(u_n**2+v_n**2+w_n**2))

    if t % 10 == 0:
        print(t+1,error,error1,error2)

1 1.0 0.14246077722568176 2.6710227054928054e-15
11 0.12177501445403775 0.12177501445403988 5.411593893995048e-14


In [7]:
#u_exact = -U*np.cos(np.pi*X0/Lx)*np.sin(np.pi*Y0/Ly)*np.exp(-2.0*np.pi**2*t_s/(Re))
#v_exact = U*np.sin(np.pi*X0/Lx)*np.cos(np.pi*Y0/Ly)*np.exp(-2.0*np.pi**2*t_s/(Re))
print(t)
u_exact = -U*np.cos(np.pi*X0/Lx)*np.sin(np.pi*Y0/Ly)*np.exp(-2.0*np.pi**2*t_s/(Re))
v_exact = U*np.sin(np.pi*X0/Lx)*np.cos(np.pi*Y0/Ly)*np.exp(-2.0*np.pi**2*t_s/(Re))
print(np.sqrt(np.sum((np.abs(u_exact[:,:,:])/U-np.abs(u[:,:,:,0])/U)**2)/((Nx+1)*(Ny+1)*(Nz+1))))
print(np.sqrt(np.sum((np.abs(v_exact[:,:,:])/U-np.abs(u[:,:,:,1])/U)**2)/((Nx+1)*(Ny+1)*(Nz+1))))
print(np.sqrt(np.sum((np.abs(u_exact[:,:,:])/U-np.abs(Psi_qlbm1[t+1,:,:,:])/U)**2)/((Nx+1)*(Ny+1)*(Nz+1))))
print(np.sqrt(np.sum((np.abs(v_exact[:,:,:])/U-np.abs(Psi_qlbm2[t+1,:,:,:])/U)**2)/((Nx+1)*(Ny+1)*(Nz+1))))

19
0.01859360271483477
0.018593602714834748
0.01859360271483632
0.018593602714836517


In [ ]:
np.savetxt("Re=10_8/QLBM_u.csv",Psi_qlbm1[t+1,:,:,:].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=10_8/QLBM_v.csv",Psi_qlbm2[t+1,:,:,:].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=10_8/QLBM_w.csv",Psi_qlbm3[t+1,:,:,:].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=10_8/QLBM_rho.csv",Psi_qlbm[t+1,:,:,:].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=10_8/CLBM_u.csv",u[:,:,:,0].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=10_8/CLBM_v.csv",u[:,:,:,1].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=10_8/CLBM_w.csv",u[:,:,:,2].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=10_8/CLBM_rho.csv",rho.reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")

In [8]:
import plotly.graph_objects as go
import matplotlib

fig = go.Figure(data=go.Isosurface(
    x=(X0/Lx).flatten(),         
    y=(Y0/Ly).flatten(),
    z=(Z0/Lz).flatten(),
    value=(u[:,:,:,1]/U).flatten(),      
    isomin=(u[:,:,:,1]/U).min(),         
    isomax=(u[:,:,:,1]/U).max(),         
    surface_count=8,        
    colorscale="jet",   
    opacity=1.0,            
    caps=dict(x_show=False, y_show=False, z_show=False)  
))
fig.update_layout(
    width=1000,
    height=1000,
    dragmode="zoom",
    autosize=True,
    font=dict(family="Times New Roman", size=16, color="black"),
    scene=dict(  
        xaxis=dict(title="x/L",title_font=dict(family="Times New Roman", style='italic', size=24)),
        yaxis=dict(title="y/L", title_font=dict(family="Times New Roman", style='italic',size=24)),
        zaxis=dict(title="z/L", title_font=dict(family="Times New Roman", style='italic',size=24)),
    ),
)
fig.update_traces(colorbar=dict(
                tickfont=dict(family="Times New Roman", size=24),    
                x = 0.95,
                y=0.45,           
                xanchor="left",  
                yanchor="middle",
                len=0.6,         
                thickness=25,  
                lenmode="fraction",  
                ))

fig.show()
#fig.write_image("plot_v_classical.png",scale=2)

In [9]:
fig = go.Figure(data=go.Isosurface(
    x=(X0/Lx).flatten(),          
    y=(Y0/Ly).flatten(),
    z=(Z0/Lz).flatten(),
    value=(Psi_qlbm2[t+1:,:,:]/U).flatten(),      
    isomin=(Psi_qlbm2[t+1:,:,:]/U).min(),         
    isomax=(Psi_qlbm2[t+1:,:,:]/U).max(),         
    surface_count=8,        
    colorscale="jet",   
    opacity=1.0,            
    caps=dict(x_show=False, y_show=False, z_show=False)  
))
fig.update_layout(
    width=1000,
    height=1000,
    dragmode="zoom",
    autosize=True,
    font=dict(family="Times New Roman", size=16, color="black"),
    scene=dict(  
        xaxis=dict(title="x/L",title_font=dict(family="Times New Roman", style='italic', size=24)),
        yaxis=dict(title="y/L", title_font=dict(family="Times New Roman", style='italic',size=24)),
        zaxis=dict(title="z/L", title_font=dict(family="Times New Roman", style='italic',size=24)),
    ),
)
fig.update_traces(colorbar=dict(
                tickfont=dict(family="Times New Roman", size=24),   
                x = 0.95,
                y=0.45,           
                xanchor="left",  
                yanchor="middle",
                len=0.6,         
                thickness=25,  
                lenmode="fraction",  
                ))

fig.show()
#fig.write_image("plot_v_quantum.png",scale=2)

In [10]:
fig = go.Figure(data=go.Isosurface(
    x=(X0/Lx).flatten(),         
    y=(Y0/Ly).flatten(),
    z=(Z0/Lz).flatten(),
    value=(v_exact/U).flatten(),    
    isomin=(v_exact/U).min(),         
    isomax=(v_exact/U).max(),         
    surface_count=8,        
    colorscale="jet",   
    opacity=1.0,           
    caps=dict(x_show=False, y_show=False, z_show=False)  
))
fig.update_layout(
    width=1000,
    height=1000,
    dragmode="zoom",
    autosize=True,
    font=dict(family="Times New Roman", size=16, color="black"),
    scene=dict(  
        xaxis=dict(title="x/L",title_font=dict(family="Times New Roman", style='italic', size=24)),
        yaxis=dict(title="y/L", title_font=dict(family="Times New Roman", style='italic',size=24)),
        zaxis=dict(title="z/L", title_font=dict(family="Times New Roman", style='italic',size=24)),
    ),
)
fig.update_traces(colorbar=dict(
                tickfont=dict(family="Times New Roman", size=24),    
                x = 0.95,
                y=0.45,          
                xanchor="left",  
                yanchor="middle",
                len=0.6,         
                thickness=25,   
                lenmode="fraction",  
                ))

fig.show()
#fig.write_image("plot_v_exact.png",scale=2)